In [2]:
!pip install langchain langchain-openai pydantic typing
print("Libraries installed successfully.")

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached annotated_types-0.7.0-py3-none-any.whl.metadata (15 kB)
  Using cached typing_inspection-0.4.2-py3-none-any.whl.metadata (2.6 kB)
   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------------- ------------------- 1.0/2.1 MB 5.9 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 5.0 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 5.0 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 5.0 MB/s eta 0:00:01
   ----------------------------------- ---- 1.8/2.1 MB 5.0 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 1.5 MB/s  0:00:01
   


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
# Install the Google Generative AI library
!pip install -q -U langchain-google-genai

import os
from getpass import getpass

# Enter your Google API Key (Get it for free at aistudio.google.com)
if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Google API Key here: ")

print("Setup complete.")


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Setup complete.


In [5]:
from pydantic import BaseModel, Field
from typing import List, Literal

# This class defines exactly what the AI must look for in each review
class SentimentResult(BaseModel):
    sentiment: Literal["Positive", "Negative", "Neutral", "Mixed"] = Field(
        description="The overall emotional category of the text."
    )
    score: int = Field(
        ge=1, le=10, description="A numerical score where 1 is angry/hateful and 10 is delighted."
    )
    detected_emotions: List[str] = Field(
        description="List of specific emotions found (e.g., 'sarcasm', 'joy', 'disappointment')."
    )
    is_sarcastic: bool = Field(
        description="Flag True if the user is being ironic or sarcastic."
    )
    analysis_reasoning: str = Field(
        description="A one-sentence explanation of why the AI chose this sentiment."
    )

print("Cell 3 complete: Sentiment Data Schema is defined and ready.")

Cell 3 complete: Sentiment Data Schema is defined and ready.


In [9]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

# Initialize the newer Gemini 2.5 model (more stable in 2026)
# If this still gives an error, try 'gemini-2.0-flash' or 'gemini-3.1-flash-preview'
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

# Connect the schema from Cell 3
structured_llm = llm.with_structured_output(SentimentResult)

# Define the instruction
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert NLP analyst. Break down the sentiment of this review."),
    ("human", "{input_text}")
])

# Create the chain
sentiment_chain = prompt | structured_llm

print("Cell 4 Updated: Using Gemini 2.5 Flash Engine.")

Cell 4 Updated: Using Gemini 2.5 Flash Engine.


In [12]:
# 1. Define a variety of test cases to explore the LM's capabilities
test_reviews = [
    "I am so happy with this purchase! The delivery was fast and the quality is amazing.",
    "The product is okay, but it arrived two weeks late. Not sure if I would buy again.",
    "Oh great, another 'upgrade' that makes the app crash every five minutes. Wonderful job, guys.",
    "The battery life is incredible, but the screen is a bit too dim for my taste."
]

# 2. Run the chain for each review and store the results
all_results = []

for review in test_reviews:
    print(f"Processing: {review[:40]}...")
    result = sentiment_chain.invoke({"input_text": review})
    all_results.append({
        "Review": review,
        "Sentiment": result.sentiment,
        "Score": result.score,
        "Sarcastic": result.is_sarcastic,
        "Emotions": ", ".join(result.detected_emotions),
        "Reasoning": result.analysis_reasoning
    })

# 3. Use Pandas to display the results in a nice table
import pandas as pd
df = pd.DataFrame(all_results)
df

Processing: I am so happy with this purchase! The de...
Processing: The product is okay, but it arrived two ...
Processing: Oh great, another 'upgrade' that makes t...
Processing: The battery life is incredible, but the ...


,Review,Sentiment,Score,Sarcastic,Emotions,Reasoning
0,I am so happy with this purchase! The delivery...,Positive,10,False,"joy, satisfaction, delight",The user explicitly states 'I am so happy' and...
1,"The product is okay, but it arrived two weeks ...",Mixed,4,False,"disappointment, hesitation",The review expresses a neutral opinion on the ...
2,"Oh great, another 'upgrade' that makes the app...",Negative,2,True,"frustration, sarcasm, annoyance, disappointment",The user employs sarcastic language like 'Oh g...
3,"The battery life is incredible, but the screen...",Mixed,7,False,"satisfaction, disappointment",The review expresses strong satisfaction with ...
